# Module 2 — Confidence-Gated Product Classification

**Retail Inventory AI · Week 4 · `retail-inventory-ai`**

This notebook is the shareable walkthrough of everything we built in Module 2. It is meant to be
read top-to-bottom by a teammate: it explains the design, shows the data, loads the trained model
and its metrics, plots the confidence-gate trade-off, and runs a live inference demo on sample crops.

## The mentor's design

Don't run an expensive VLM on **every** shelf crop. Instead:

1. **Label cheaply** — get product labels for a sample of crops with a VLM (Gemini), *once*.
2. **Train a classifier** on those teacher labels.
3. **Gate on confidence at inference** — if the classifier is confident, use it (fast, free);
   if not, fall back to the VLM for just that one crop.

So the classifier carries the easy majority and the VLM is reserved for the genuinely hard crops.
The **fallback rate** (fraction routed to the VLM) is a headline cost metric.

## The key finding that shaped everything

The shared val label CSV has a `product_label` column with **9,685 unique values** (~1.5 examples
each) — that is why the earlier attempt to classify product *names* scored near-zero. It is not a
learnable target. So our classifier targets the **normalized taxonomy** instead:

| Target | Classes | Trainable? |
|---|---|---|
| `product_label` (fine name) | 9,685 (~1.5 ex each) | ❌ |
| **`normalized_category`** | **18** | ✅ |
| **`normalized_subcategory`** | **48** | ✅ |

The fine product name is left to the VLM fallback only. The classifier is a **two-head** model
(one 18-way category head, one 48-way subcategory head) over this fixed taxonomy.


## 0 · Setup

All paths are relative to the repo root. The notebook reads the committed artifacts in
`classification/artifacts/clip_v1/` — no GCS access required. Heavy deps (`torch`, `open_clip`)
are only needed for the **inference demo** (section 5); the results/plots sections need just
`pandas` + `matplotlib`.

```bash
# from the repo root, if you want to run the inference demo:
pip install torch torchvision open-clip-torch pandas matplotlib pillow
# and fetch the LFS-tracked weights:
git lfs install && git lfs pull
```


In [ ]:
import json
from pathlib import Path

# Resolve the repo root whether the notebook runs from notebooks/ or the repo root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ART = ROOT / "classification" / "artifacts" / "clip_v1"
assert ART.exists(), f"artifacts not found at {ART} — run `git lfs pull`?"

metrics    = json.loads((ART / "metrics.json").read_text())
gate_sweep = json.loads((ART / "gate_sweep.json").read_text())
label_maps = json.loads((ART / "label_maps.json").read_text())

CATEGORIES    = label_maps["category_names"]
SUBCATEGORIES = label_maps["subcategory_names"]
print(f"{len(CATEGORIES)} categories, {len(SUBCATEGORIES)} subcategories")
print("backbone:", metrics["backbone"])

## 1 · The taxonomy (18 categories / 48 subcategories)

Single source of truth in `classification/taxonomy.py` (regenerated from the shared val CSV).
Both the labeler and the classifier use exactly these strings, so nothing drifts.


In [ ]:
print("CATEGORIES (18):")
for c in CATEGORIES:
    print("  -", c)
print(f"\nSUBCATEGORIES (48): {', '.join(SUBCATEGORIES)}")

## 2 · Crops + teacher labels

**Crops.** We ran our YOLOv11 detector (`detection/artifacts/v11/best.pt`, mAP@0.5 ≈ 0.92) over
SKU-110K shelf images and cropped every detected box. At `--fraction 0.10` (10% of train, all of
val) that produced **127,706 train** and **92,597 val** crops. They live as single tarballs in GCS
(`crops/{train,val}_crops.tar`) — tarring before upload was ~100× faster than uploading 600k+ tiny
files individually.

**Teacher labels (Gemini).** We labeled a **30,000-crop** sample of the train crops with Vertex AI
`gemini-2.5-flash`, constrained to the 48-subcategory enum via a JSON response schema, with
`thinking_budget=0` (single-label classification needs no chain-of-thought; this cut ~34% of token
spend). Result: **30,000 labeled, 0 errors, 27.9M tokens, 779 s, 38.5 crops/s** (32-worker CPU VM).

> **Why Gemini and not CLIP?** We tried CLIP zero-shot as the (free) labeler first and rejected it:
> on a visual spot-check it got 1/4 vs Gemini's 4/4, and the two agreed on only 24% of categories.
> CLIP is too weak on small/partial shelf crops. (See `RESULTS.md` for the decision-gate detail.)

The category mix below shows ~44% of crops are **"Other / Unclear"** — tiny, blurry, or
back-of-pack crops with no identifiable branding. That is honest and expected for SKU-110K, and it
is exactly the population the confidence gate should route to the VLM.


In [ ]:
import pandas as pd

# Train-label category distribution (measured on the 30k Gemini-labeled train crops).
train_dist = {
    "Other / Unclear": 13318, "Personal Care & Beauty": 5227, "Beverages": 3436,
    "Health & Wellness": 2461, "Household Cleaning & Laundry": 1512, "Oral Care": 1112,
    "Hair Care": 1093, "Paper & Hygiene": 778, "Home & Kitchen": 289, "Baby Care": 257,
    "Grocery & Pantry": 181, "Snacks & Confectionery": 170, "Automotive & Hardware": 61,
    "Tobacco / Restricted": 45, "Apparel & Accessories": 27, "Stationery & Office": 15,
    "Pet Care": 14, "Electronics": 4,
}
N = sum(train_dist.values())
df = pd.DataFrame({"category": list(train_dist), "count": list(train_dist.values())})
df["pct"] = (100 * df["count"] / N).round(1)
print(f"30,000 Gemini-labeled train crops · all 48 subcategories represented")
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
d = df.sort_values("count")
ax.barh(d["category"], d["count"], color="#4C78A8")
ax.set_xlabel("crops (of 30,000)")
ax.set_title("Gemini teacher-label category distribution (train sample)")
for i, (c, p) in enumerate(zip(d["count"], d["pct"])):
    ax.text(c + 80, i, f"{p}%", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 3 · The classifier and its results

**Architecture** (`classification/classifier_lib.py`): a frozen **open-clip ViT-B-32** image
encoder feeding **two linear heads** (18-way category + 48-way subcategory). Only the heads train.
At inference the subcategory head is **masked to the children of the predicted category**, so the
two outputs are always consistent (the taxonomy is a clean tree).

Trained 15 epochs on one L4 GPU; evaluated on **2,000 held-out Gemini-labeled val crops** (our own
crops, so the filename join is valid — unlike the cross-detector join to the teammate's CSV).


In [ ]:
best = metrics["best"]
print("Best epoch:", best["epoch"])
pd.DataFrame([{
    "Category top-1":    f"{best['cat_top1']:.1%}",
    "Category top-3":    f"{best['cat_top3']:.1%}",
    "Subcategory top-1": f"{best['sub_top1']:.1%}",
    "Cat macro-F1":      f"{best['cat_macro_f1']:.3f}",
    "Sub macro-F1":      f"{best['sub_macro_f1']:.3f}",
}]).T.rename(columns={0: "value"})

In [ ]:
# Training history — accuracy per epoch.
hist = pd.DataFrame(metrics["history"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(hist["epoch"], hist["cat_top1"], "-o", label="cat top-1")
a1.plot(hist["epoch"], hist["cat_top3"], "-o", label="cat top-3")
a1.plot(hist["epoch"], hist["sub_top1"], "-o", label="subcat top-1")
a1.axvline(best["epoch"], color="grey", ls="--", lw=1, label=f"best (ep {best['epoch']})")
a1.set_xlabel("epoch"); a1.set_ylabel("accuracy"); a1.set_title("Validation accuracy"); a1.legend(); a1.grid(alpha=.3)
a2.plot(hist["epoch"], hist["train_loss"], "-o", color="#E45756")
a2.set_xlabel("epoch"); a2.set_ylabel("train loss"); a2.set_title("Training loss"); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Reading the numbers.** Category top-1 of **60%** (top-3 **85%**) and subcategory top-1 of
**52%** decisively beat the earlier near-zero product-name attempt — confirming the 18/48 taxonomy
is learnable. The macro-F1 is low (0.25 / 0.19) because it averages every class equally and the
rare classes barely have training examples (Electronics = 4, Pet Care = 14). Accuracy plateaus
early — the expected ceiling of a *frozen* backbone (linear probe). A ResNet-50 fine-tune
(`BACKBONE=resnet50`) is the planned next experiment to push past it.


## 4 · The confidence gate

This is the core deliverable. We sweep the confidence threshold over the 2,000 val crops and, at
each threshold, measure: **accept rate** (crops the classifier keeps), **fallback rate** (crops sent
to the VLM), and **accuracy on the accepted crops**. `overall_cat (ub)` is an upper bound on system
accuracy assuming a perfect VLM on the fallback set.

The curve is **monotonic and well-calibrated** — the classifier's confidence genuinely tracks its
correctness, which is what makes the gate work.


In [ ]:
g = pd.DataFrame(gate_sweep)
show = g[g["threshold"].isin([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])].copy()
for c in ["accept_rate", "fallback_rate", "acc_accepted_cat", "acc_accepted_sub", "overall_cat_upperbound"]:
    show[c] = (100 * show[c]).round(1).astype(str) + "%"
show.rename(columns={
    "threshold": "Threshold", "accept_rate": "Accept", "fallback_rate": "Fallback",
    "acc_accepted_cat": "Acc@accept (cat)", "acc_accepted_sub": "Acc@accept (sub)",
    "overall_cat_upperbound": "Overall cat (ub)",
}).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(g["accept_rate"] * 100, g["acc_accepted_cat"] * 100, "-o", label="category acc @ accepted")
ax.plot(g["accept_rate"] * 100, g["acc_accepted_sub"] * 100, "-o", label="subcategory acc @ accepted")
ax.axhline(metrics["best"]["cat_top1"] * 100, color="grey", ls=":", lw=1, label="ungated cat top-1 (60.2%)")

# Mark the recommended operating point (threshold 0.60).
op = g[g["threshold"] == 0.6].iloc[0]
ax.scatter([op["accept_rate"] * 100], [op["acc_accepted_cat"] * 100], s=160,
           facecolors="none", edgecolors="red", linewidths=2, zorder=5)
ax.annotate(f"  thr=0.60\n  accept {op['accept_rate']:.0%}, cat {op['acc_accepted_cat']:.0%}",
            (op["accept_rate"] * 100, op["acc_accepted_cat"] * 100), color="red", fontsize=9)
ax.set_xlabel("Accept rate  (% handled by classifier — rest go to VLM)")
ax.set_ylabel("Accuracy on accepted crops (%)")
ax.set_title("Confidence-gate trade-off: coverage vs accuracy")
ax.legend(); ax.grid(alpha=.3); ax.invert_xaxis()
plt.tight_layout(); plt.show()

**Recommended operating point: threshold 0.60.** The classifier confidently handles ~half the
crops at **78.5% category / 70.1% subcategory** accuracy (vs 60%/52% if it had to label everything),
and the hard half goes to Gemini. Tune to taste: **0.70** → ~85% on the accepted third;
**0.50** → ~74% on two-thirds.


## 5 · Live inference demo

Load the trained classifier and run the **confidence gate** on sample crops in
`notebooks/samples_crops/`. For each crop we print the predicted category/subcategory, the
confidence, and the gate decision at threshold 0.60 (`classifier` vs `→ VLM fallback`).

> Requires `torch`, `open-clip-torch`, `pillow` and the LFS weights (`git lfs pull`). The first run
> downloads the open-clip ViT-B-32 weights (~600 MB) to build the frozen backbone.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))   # so `classification` + `autolabel` import

import torch
from PIL import Image
from classification import taxonomy, classifier_lib as lib

THRESHOLD = 0.60
device = "cuda" if torch.cuda.is_available() else "cpu"

tax = taxonomy.load()
ckpt = torch.load(ART / "classifier.pt", map_location=device)
model = lib.make_model(ckpt["backbone"], tax).to(device)
model.load_state_dict(ckpt["state_dict"]); model.eval()
preprocess = model.preprocess
print(f"loaded {ckpt['backbone']} classifier on {device}; gate threshold = {THRESHOLD}")

In [ ]:
samples = sorted((ROOT / "notebooks" / "samples_crops").rglob("*.jpg"))
print(f"{len(samples)} sample crops\n")

rows = []
with torch.no_grad():
    for p in samples:
        x = preprocess(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        cat_logits, sub_logits = model(x)
        cid, cconf, sid, _ = lib.masked_subcategory_pred(cat_logits, sub_logits, tax)
        conf = float(cconf[0])
        rows.append({
            "crop": p.name,
            "pred_category": tax.category_names[int(cid[0])],
            "pred_subcategory": tax.subcategory_names[int(sid[0])],
            "confidence": round(conf, 3),
            "route": "classifier" if conf >= THRESHOLD else "→ VLM fallback",
        })
import pandas as pd
pd.DataFrame(rows)

In [ ]:
# Show the crops with their predictions side by side.
import matplotlib.pyplot as plt
from PIL import Image

n = len(samples)
cols = 3; r = (n + cols - 1) // cols
fig, axes = plt.subplots(r, cols, figsize=(12, 4 * r))
for ax, row, p in zip(axes.ravel(), rows, samples):
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    tag = "✓ clf" if row["route"] == "classifier" else "→ VLM"
    ax.set_title(f"{row['pred_category']}\n{row['pred_subcategory']}\nconf={row['confidence']:.2f}  [{tag}]",
                 fontsize=8)
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 6 · How to reproduce / run the pipeline

All runs happen on GCP (no-public-IP VMs, Cloud NAT egress, self-deleting). Launchers are local;
they package the code, create the VM, and the VM self-deletes when it syncs results to GCS.

```bash
source .env   # PROJECT_ID, REGION, ZONE, BUCKET, VERTEX_MODEL=gemini-2.5-flash

# 1. Crops (10% of train) — already done; crops/{train,val}_crops.tar in GCS.
bash classification/launch_crops.sh

# 2. Teacher-label a 30k train sample with Gemini.
SPLIT=train LABEL_LIMIT=30000 WORKERS=32 bash autolabel/launch_labels.sh

# 3. Train the two-head classifier (CLIP-frozen baseline).
export TRAIN_LABELS=$BUCKET/labels/labels_gemini_train.csv
export VAL_LABELS=$BUCKET/labels/labels_gemini_val.csv
BACKBONE=clip VARIANT=clip_v1 EPOCHS=15 bash classification/launch_train.sh

# 4. Tune the confidence gate on the held-out val crops.
VARIANT=clip_v1 SPLIT=val bash classification/launch_tune_gate.sh

# Artifacts land in gs://<bucket>/classifier/clip_v1/ and are mirrored in
# classification/artifacts/clip_v1/ (this notebook reads them).
```

### Files
| Path | Role |
|---|---|
| `classification/taxonomy.py` | 18/48 taxonomy — single source of truth |
| `classification/gen_crops.py` | detector → per-product crops |
| `autolabel/label_vlm.py` | Gemini teacher-labeler (JSON-schema constrained) |
| `classification/classifier_lib.py` | two-head model + dataset |
| `classification/train_classifier.py` | training loop |
| `classification/infer.py` | confidence-gated inference + `--tune-threshold` |
| `classification/artifacts/clip_v1/` | trained weights + metrics + gate sweep (this notebook) |

**Next:** ResNet-50 fine-tune variant for comparison; scale labeling to all 127k train crops;
full shelf-image demo (detect → classify → fallback).
